In [8]:
# Point Negation
P=(8045,6936)
Q = (P[0], -P[1] % 9739)
Q

(8045, 2803)

In [ ]:
# Point Addition

def add(x, y, a, p):
    x1, y1 = x
    x2, y2 = y
    if x1 == 0 and y1 == 0:
        return x2, y2
    if x2 == 0 and y2 == 0:
        return x1, y1
    if x1 == x2 and y1 == -y2:
        return 0, 0

    if x1 == x2 and y1 == y2:
        lam = (3 * x1 * x1 + a) * pow(2 * y1, -1, p) % p
    else:
        lam = (y2 - y1) * pow(x2 - x1, -1, p) % p

    x3 = (lam * lam - x1 - x2) % p
    y3 = (lam * (x1 - x3) - y1) % p

    return x3, y3

# Eliptic curve: Y^2 = X^3 + 497X + 1768 mod 9739

P=(493,5564)
Q=(1539,4742)
R=(4403,5202)

PP = add(P, P, 497, 9739)
PPQ = add(PP, Q, 497, 9739)
PPQR = add(PPQ, R, 497, 9739)
print(PPQR)
x = PPQR[0]
y = PPQR[1]
assert y**2 % 9739 == (x**3 + 497*x + 1768) % 9739

(4215, 2162)


In [ ]:
# Scalar Multiplication

def scalar_mul(v,n,a,p):
    x,y = v
    result = (0,0)
    power = (x,y)
    while n > 0:
        if n & 1:
            result = add(result, power, a, p)
        power = add(power, power, a, p)
        n >>= 1
    return result

# Eliptic curve: Y^2 =X^3 + 497X + 1768 mod 9739

P=(2339,2213)
scalar_mul(P, 7863, 497, 9739)

(9467, 2742)

In [ ]:
# Curves and Logs

from Crypto.Hash import SHA1

# Eliptic curve: Y^2 = X^3 + 497X + 1768 mod 9739

Q_A = (815,3190)
n_B = 1829
S = scalar_mul(Q_A, n_B, 497, 9739)
h = SHA1.new()
h.update(str(S[0]).encode())
print(h.hexdigest())

80e5212754a824d3a4aed185ace4f9cac0f908bf


In [ ]:
# Efficient Exchange

from Crypto.Cipher import AES
from Crypto.Util.Padding import pad, unpad
import hashlib


def is_pkcs7_padded(message):
    padding = message[-message[-1]:]
    return all(padding[i] == len(padding) for i in range(0, len(padding)))

# from EfficientExchange/decrypt.py
def decrypt_flag(shared_secret: int, iv: str, ciphertext: str):
    # Derive AES key from shared secret
    sha1 = hashlib.sha1()
    sha1.update(str(shared_secret).encode('ascii'))
    key = sha1.digest()[:16]
    # Decrypt flag
    ciphertext = bytes.fromhex(ciphertext)
    iv = bytes.fromhex(iv)
    cipher = AES.new(key, AES.MODE_CBC, iv)
    plaintext = cipher.decrypt(ciphertext)

    if is_pkcs7_padded(plaintext):
        return unpad(plaintext, 16).decode('ascii')
    else:
        return plaintext.decode('ascii')

iv = "cd9da9f1c60925922377ea952afc212c"
encrypted_flag = "febcbe3a3414a730b125931dccf912d2239f3e969c4334d95ed0ec86f6449ad8"

# Eliptic curve: Y^2 = X^3 + 497X + 1768 mod 9739

x_Q_A = 4726
n_B = 6534
y_Q_A_square = (x_Q_A**3 + 497*x_Q_A + 1768) % 9739

# in both cases [n](x,y) and [n](x,-y) give the same x coordinate, so no matter which y we pick, the shared secret x coordinate will be the same

for y_Q_A in range(9739):
    if (y_Q_A * y_Q_A) % 9739 == y_Q_A_square:
        shared_secret = scalar_mul((x_Q_A, y_Q_A), n_B, 497, 9739)
        flag = decrypt_flag(shared_secret[0], iv, encrypted_flag)
        print(flag,shared_secret[0])

crypto{3ff1c1ent_k3y_3xch4ng3} 1791
crypto{3ff1c1ent_k3y_3xch4ng3} 1791
